In [1]:
# ============================================================
# Quant ML Pipeline — S&P 500 & KOSPI
# Seoul National University | Seongwon Song
# ============================================================

import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# 1. Data Collection
# ============================================================

url = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv"
df_sp500 = pd.read_csv(url)
tickers = df_sp500["Symbol"].tolist()
sub_industry_map = df_sp500.set_index("Symbol")["GICS Sub-Industry"].to_dict()

data = yf.download(tickers, start="2015-01-01", end="2026-08-29", auto_adjust=True)["Close"]
volume = yf.download(tickers, start="2015-01-01", end="2026-08-29", auto_adjust=True)["Volume"]
data = data.dropna(axis=1, thresh=int(len(data) * 0.8))
volume = volume[data.columns]

sp500 = yf.download("^GSPC", start="2015-01-01", end="2026-08-29", auto_adjust=True)["Close"].squeeze()
vix_series = yf.download("^VIX", start="2015-01-01", end="2026-08-29", auto_adjust=True)["Close"].squeeze()

print(f"Tickers: {len(data.columns)}")
print(f"Date range: {data.index[0].date()} ~ {data.index[-1].date()}")

[*************         27%                       ]  138 of 503 completed$BF.B: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-08-29)
[*****************     36%                       ]  180 of 503 completed$LEN: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-08-29)
[*****************     36%                       ]  181 of 503 completed$MMM: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-08-29)
[*****************     36%                       ]  182 of 503 completed$GL: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-08-29)
[*****************     36%                       ]  183 of 503 completed$HST: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-08-29)
[******************    37%                       ]  184 of 503 completed$VZ: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-08-29)
[******************    37%                       ]  185 of 503 completed$NI: possibly delisted; no price data fou

Tickers: 462
Date range: 2015-01-02 ~ 2026-08-28


In [2]:
# ============================================================
# 2. Feature Engineering
# ============================================================

features = pd.DataFrame()

for ticker in data.columns:
    close = data[ticker]
    vol = volume[ticker]
    
    temp = pd.DataFrame(index=close.index)
    temp["ma20"] = close.rolling(20).mean()
    temp["ma50"] = close.rolling(50).mean()
    temp["golden_cross"] = temp["ma20"] / temp["ma50"]
    temp["mom1w"] = close.pct_change(5)
    temp["mom1m"] = close.pct_change(21)
    temp["mom3m"] = close.pct_change(63)
    
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    temp["RSI"] = 100 - (100 / (1 + gain / loss))
    
    temp["MACD"] = (close.rolling(12).mean() - close.rolling(26).mean()) / close
    temp["vol_change"] = vol.pct_change(5)
    temp["vol_vs_avg"] = vol / vol.rolling(21).mean()
    
    ma20_slope = temp["ma20"].diff() / close
    temp["ma20_curvature"] = ma20_slope.diff() / close
    
    temp["target_short"] = (close.shift(-5) > close).astype(int)
    temp["target_long"] = (close.shift(-21) > close).astype(int)
    temp["ticker"] = ticker
    
    features = pd.concat([features, temp])

features = features.replace([np.inf, -np.inf], np.nan)
ticker_col = features["ticker"].copy()
features = features.groupby("ticker").ffill().bfill()
features["ticker"] = ticker_col
features = features.dropna()

print(f"Features shape: {features.shape}")

Features shape: (1354122, 14)


In [6]:
# ============================================================
# 3. Market Regime Detection & Sector Rotation
# ============================================================

def get_market_regime(date, sp500, vix_series, bull_score=3, bear_score=1):
    date = pd.Timestamp(date)
    if date not in sp500.index:
        return "unknown"
    idx = sp500.index.get_loc(date)
    if idx < 50:
        return "unknown"
    sp500_now = sp500.iloc[idx]
    ma20 = sp500.iloc[max(0, idx-20):idx].mean()
    ma50 = sp500.iloc[max(0, idx-50):idx].mean()
    ret_1w = sp500.iloc[idx] / sp500.iloc[idx-5] - 1
    ret_1m = sp500.iloc[idx] / sp500.iloc[idx-21] - 1
    vix_now = vix_series[date] if date in vix_series.index else 20
    score = 0
    if sp500_now > ma50: score += 1
    if sp500_now > ma20: score += 1
    if vix_now < 20:     score += 1
    if ret_1w > 0:       score += 1
    if ret_1m > 0:       score += 1
    if score >= bull_score:   return "bull"
    elif score <= bear_score: return "bear"
    else:                     return "neutral"

def get_top_sub_industries_v2(date, data, sub_industry_map, n=5):
    if date not in data.index:
        return []
    idx = data.index.get_loc(date)
    if idx < 21:
        return []
    ret_1w = data.iloc[idx] / data.iloc[max(0, idx-5)] - 1
    ret_1m = data.iloc[idx] / data.iloc[max(0, idx-21)] - 1
    sector_scores = {}
    for ticker in data.columns:
        sector = sub_industry_map.get(ticker)
        if sector is None:
            continue
        if sector not in sector_scores:
            sector_scores[sector] = []
        score = 0.4 * ret_1w[ticker] + 0.6 * ret_1m[ticker]
        sector_scores[sector].append(score)
    sector_avg = {s: np.mean(r) for s, r in sector_scores.items() if len(r) >= 3}
    return sorted(sector_avg, key=sector_avg.get, reverse=True)[:n]

# Current market status
latest_date = features.index.max()
regime = get_market_regime(latest_date, sp500, vix_series)
print(f"Date: {latest_date.date()}")
print(f"Regime: {regime}")
print(f"Top sectors: {get_top_sub_industries_v2(latest_date, data, sub_industry_map)}")

Date: 2026-08-28
Regime: bear
Top sectors: ['Life Sciences Tools & Services', 'Technology Hardware, Storage & Peripherals', 'Biotechnology', 'Health Care Equipment', 'Property & Casualty Insurance']


In [7]:
# ============================================================
# 4. RF Backtesting v12 (Walk-Forward Validation)
# ============================================================

feature_cols_bull = ["mom1w", "MACD", "vol_change", 
                     "vol_vs_avg", "ma20_curvature",
                     "golden_cross", "RSI"]
feature_cols_long = ["mom1m", "mom3m", "MACD", 
                     "vol_vs_avg", "RSI", "golden_cross"]

def backtest_v12(features, data, test_years=range(2022, 2026),
                 take_profit=0.05, stop_loss=-0.05,
                 bull_score=3, bear_score=1):
    
    all_returns = []
    
    for test_year in test_years:
        print(f"Testing {test_year}...")
        
        train_wf = features[features.index < f"{test_year}-01-01"]
        test_wf = features[(features.index >= f"{test_year}-01-01") & 
                           (features.index < f"{test_year+1}-01-01")]
        data_test = data
        
        sp500_wf = sp500[sp500.index < f"{test_year}-01-01"]
        vix_wf = vix_series[vix_series.index < f"{test_year}-01-01"]
        regime_wf = pd.Series(
            [get_market_regime(d, sp500_wf, vix_wf,
                               bull_score=bull_score, bear_score=bear_score)
             for d in sp500_wf.index[50:]], index=sp500_wf.index[50:])
        
        bull_idx = train_wf.index.isin(regime_wf[regime_wf == "bull"].index)
        
        m_bull = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)
        m_bull.fit(train_wf[bull_idx][feature_cols_bull], train_wf[bull_idx]["target_short"])
        m_long = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)
        m_long.fit(train_wf[feature_cols_long], train_wf["target_long"])
        
        COMMISSION = 0.002
        N = 3
        cash = 1.0
        shares = {}
        buy_prices = {}
        next_buy = set()
        sell_set = set()
        history = []
        
        all_dates = sorted(test_wf.index.unique())
        
        for date in all_dates[:-1]:
            today = test_wf[test_wf.index == date].reset_index(drop=True)
            if len(today) < 10: continue
            
            act_date = data_test.index[data_test.index <= date][-1]
            sp500_now = sp500[sp500.index <= date]
            vix_now = vix_series[vix_series.index <= date]
            regime = get_market_regime(date, sp500_now, vix_now,
                                       bull_score=bull_score, bear_score=bear_score)
            
            # Daily: stop loss
            for t in list(shares.keys()):
                if t not in data_test.columns: continue
                ret = data_test.loc[act_date, t] / buy_prices[t] - 1
                if ret < stop_loss:
                    cash += shares.pop(t) * data_test.loc[act_date, t] * (1 - COMMISSION)
                    buy_prices.pop(t)
            
            # Thursday: decide sell/buy
            if date.weekday() == 3:
                if regime == "bear":
                    sell_set = set(shares.keys())
                else:
                    sell_set = set()
                    for t in list(shares.keys()):
                        if t not in data_test.columns: continue
                        ret = data_test.loc[act_date, t] / buy_prices[t] - 1
                        if ret >= take_profit:
                            sell_set.add(t)
                
                top_subs = get_top_sub_industries_v2(date, data_test, sub_industry_map, n=5)
                today["sub"] = today["ticker"].map(sub_industry_map)
                today_bull = today[today["sub"].isin(top_subs)].dropna(subset=feature_cols_bull).copy()
                if len(today_bull) >= N:
                    today_bull["prob"] = m_bull.predict_proba(today_bull[feature_cols_bull])[:, 1]
                    new_picks = set(today_bull.nlargest(N, "prob")["ticker"])
                else:
                    new_picks = set()
                
                hold_set = (set(shares.keys()) - sell_set) & new_picks
                n_to_buy = N - len(hold_set)
                next_buy = set(list(new_picks - hold_set)[:n_to_buy])
            
            # Friday: execute
            elif date.weekday() == 4:
                for t in list(sell_set):
                    if t in shares and t in data_test.columns:
                        cash += shares.pop(t) * data_test.loc[act_date, t] * (1 - COMMISSION)
                        buy_prices.pop(t, None)
                
                if regime == "bear":
                    for t in list(shares.keys()):
                        if t in data_test.columns:
                            cash += shares.pop(t) * data_test.loc[act_date, t] * (1 - COMMISSION)
                            buy_prices.pop(t, None)
                else:
                    total_asset = cash + sum(
                        shares[t] * data_test.loc[act_date, t]
                        for t in shares if t in data_test.columns
                    )
                    per = total_asset / N
                    for t in next_buy:
                        if t not in data_test.columns: continue
                        price = data_test.loc[act_date, t]
                        shares[t] = (per / (1 + COMMISSION)) / price
                        buy_prices[t] = price
                        cash -= per / (1 + COMMISSION)
                
                next_buy = set()
                sell_set = set()
            
            total = cash + sum(
                shares[t] * data_test.loc[act_date, t]
                for t in shares if t in data_test.columns
            )
            history.append({"date": date, "year": test_year, "total": total})
        
        if history:
            yr_ret = history[-1]["total"] / history[0]["total"] - 1
            all_returns.extend(history)
            print(f"  {test_year}: {yr_ret:.2%}")
    
    return all_returns

results = backtest_v12(features, data, take_profit=0.05)

sp500_rets = {2022: -0.1995, 2023: 0.2473, 2024: 0.2401, 2025: 0.1665}
print("\n=== Backtest Results ===")
cum = 1.0
for yr in range(2022, 2026):
    yr_data = [h for h in results if h["year"] == yr]
    if yr_data:
        yr_ret = yr_data[-1]["total"] / yr_data[0]["total"] - 1
        cum *= (1 + yr_ret)
        print(f"{yr}: Model {yr_ret:.2%} | S&P 500 {sp500_rets[yr]:.2%} | Cumulative {cum:.4f}")

Testing 2022...
  2022: -32.92%
Testing 2023...
  2023: 30.17%
Testing 2024...
  2024: 48.76%
Testing 2025...
  2025: 18.29%

=== Backtest Results ===
2022: Model -32.92% | S&P 500 -19.95% | Cumulative 0.6708
2023: Model 30.17% | S&P 500 24.73% | Cumulative 0.8731
2024: Model 48.76% | S&P 500 24.01% | Cumulative 1.2988
2025: Model 18.29% | S&P 500 16.65% | Cumulative 1.5364


In [8]:
# ============================================================
# 5. LSTM Screening (Real-time)
# ============================================================

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

SEQ_LEN = 20

def make_sequences(features, feature_cols, seq_len=20):
    X, y, meta = [], [], []
    for ticker in features["ticker"].unique():
        df = features[features["ticker"] == ticker].sort_index()
        df = df.dropna(subset=feature_cols + ["target_short"])
        vals = df[feature_cols].values
        targets = df["target_short"].values
        dates = df.index
        for i in range(seq_len, len(vals)):
            X.append(vals[i-seq_len:i])
            y.append(targets[i])
            meta.append((dates[i], ticker))
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), meta

print("Building sequences...")
X_all, y_all, meta_all = make_sequences(features, feature_cols_bull)
print(f"Shape: {X_all.shape}")

model_lstm = Sequential([
    LSTM(32, input_shape=(SEQ_LEN, len(feature_cols_bull))),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid")
])
model_lstm.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model_lstm.fit(X_all, y_all, epochs=5, batch_size=128, validation_split=0.1, verbose=1)

Building sequences...
Shape: (1344882, 20, 7)
Epoch 1/5
9457/9457 ━━━━━━━━━━━━━━━━━━━━ 223s 23ms/step - accuracy: 0.5443 - loss: 0.6892 - val_accuracy: 0.5392 - val_loss: 0.6900
Epoch 2/5
9457/9457 ━━━━━━━━━━━━━━━━━━━━ 208s 22ms/step - accuracy: 0.5446 - loss: 0.6890 - val_accuracy: 0.5423 - val_loss: 0.6895
Epoch 3/5
9457/9457 ━━━━━━━━━━━━━━━━━━━━ 1232s 130ms/step - accuracy: 0.5462 - loss: 0.6882 - val_accuracy: 0.5438 - val_loss: 0.6874
Epoch 4/5
9457/9457 ━━━━━━━━━━━━━━━━━━━━ 169s 18ms/step - accuracy: 0.5464 - loss: 0.6878 - val_accuracy: 0.5439 - val_loss: 0.6875
Epoch 5/5
9457/9457 ━━━━━━━━━━━━━━━━━━━━ 168s 18ms/step - accuracy: 0.5468 - loss: 0.6875 - val_accuracy: 0.5442 - val_loss: 0.6871


In [9]:
# ============================================================
# 6. Weekly Screening (RF → LSTM → top3)
# ============================================================

# Train RF on all bull data
regime_all = pd.Series(
    [get_market_regime(d, sp500, vix_series) for d in sp500.index[50:]],
    index=sp500.index[50:]
)
bull_dates_all = regime_all[regime_all == "bull"].index
train_bull_all = features[features.index.isin(bull_dates_all)].dropna(subset=feature_cols_bull)

model_rf = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)
model_rf.fit(train_bull_all[feature_cols_bull], train_bull_all["target_short"])

def weekly_screening(features, data, model_rf, model_lstm,
                     sp500, vix_series, sub_industry_map,
                     feature_cols, seq_len=20):
    
    latest_date = features.index.max()
    print(f"Screening date: {latest_date.date()}")
    
    sp500_now = sp500[sp500.index <= latest_date]
    vix_now = vix_series[vix_series.index <= latest_date]
    regime = get_market_regime(latest_date, sp500_now, vix_now)
    print(f"Market regime: {regime}")
    
    if regime == "bear":
        print("Bear market — hold cash")
        return []
    
    today = features[features.index == latest_date].reset_index(drop=True)
    top_subs = get_top_sub_industries_v2(latest_date, data, sub_industry_map, n=5)
    print(f"Top sectors: {top_subs}")
    
    today["sub"] = today["ticker"].map(sub_industry_map)
    today_f = today[today["sub"].isin(top_subs)].dropna(subset=feature_cols).copy()
    
    # RF: top10
    today_f["rf_prob"] = model_rf.predict_proba(today_f[feature_cols])[:, 1]
    top10 = today_f.nlargest(10, "rf_prob")
    print(f"\nRF top10: {list(top10['ticker'])}")
    
    # LSTM: top10 → top3
    lstm_probs = []
    for _, row in top10.iterrows():
        ticker = row["ticker"]
        ticker_data = features[
            (features["ticker"] == ticker) &
            (features.index <= latest_date)
        ].tail(seq_len).dropna(subset=feature_cols)
        
        if len(ticker_data) < seq_len:
            lstm_probs.append(0)
            continue
        
        seq = ticker_data[feature_cols].values[-seq_len:]
        seq = seq.reshape(1, seq_len, len(feature_cols))
        prob = model_lstm.predict(seq, verbose=0)[0][0]
        lstm_probs.append(float(prob))
    
    top10["lstm_prob"] = lstm_probs
    top3 = top10.nlargest(3, "lstm_prob")
    
    print(f"\n=== Weekly Top3 ===")
    for _, row in top3.iterrows():
        print(f"{row['ticker']}: RF {row['rf_prob']:.3f} | LSTM {row['lstm_prob']:.3f}")
    
    return list(top3["ticker"])

top3 = weekly_screening(features, data, model_rf, model_lstm,
                        sp500, vix_series, sub_industry_map, feature_cols_bull)

Screening date: 2026-08-28
Market regime: bear
Bear market — hold cash


In [10]:
# Bear 이지만 스크리닝
def weekly_screening_force(features, data, model_rf, model_lstm,
                     sp500, vix_series, sub_industry_map,
                     feature_cols, seq_len=20):
    
    latest_date = features.index.max()
    print(f"Screening date: {latest_date.date()}")
    
    regime = get_market_regime(latest_date, sp500, vix_series)
    print(f"Market regime: {regime} (screening forced)")
    
    today = features[features.index == latest_date].reset_index(drop=True)
    top_subs = get_top_sub_industries_v2(latest_date, data, sub_industry_map, n=5)
    print(f"Top sectors: {top_subs}")
    
    today["sub"] = today["ticker"].map(sub_industry_map)
    today_f = today[today["sub"].isin(top_subs)].dropna(subset=feature_cols).copy()
    
    # RF: top10
    today_f["rf_prob"] = model_rf.predict_proba(today_f[feature_cols])[:, 1]
    top10 = today_f.nlargest(10, "rf_prob")
    print(f"\nRF top10: {list(top10['ticker'])}")
    
    # LSTM: top10 → top3
    lstm_probs = []
    for _, row in top10.iterrows():
        ticker = row["ticker"]
        ticker_data = features[
            (features["ticker"] == ticker) &
            (features.index <= latest_date)
        ].tail(seq_len).dropna(subset=feature_cols)
        
        if len(ticker_data) < seq_len:
            lstm_probs.append(0)
            continue
        
        seq = ticker_data[feature_cols].values[-seq_len:]
        seq = seq.reshape(1, seq_len, len(feature_cols))
        prob = model_lstm.predict(seq, verbose=0)[0][0]
        lstm_probs.append(float(prob))
    
    top10["lstm_prob"] = lstm_probs
    top3 = top10.nlargest(3, "lstm_prob")
    
    print(f"\n=== Weekly Top3 (Bear Market — Use with Caution) ===")
    for _, row in top3.iterrows():
        print(f"{row['ticker']}: RF {row['rf_prob']:.3f} | LSTM {row['lstm_prob']:.3f}")
    
    return list(top3["ticker"])

top3 = weekly_screening_force(features, data, model_rf, model_lstm,
                              sp500, vix_series, sub_industry_map, feature_cols_bull)

Screening date: 2026-08-28
Market regime: bear (screening forced)
Top sectors: ['Life Sciences Tools & Services', 'Technology Hardware, Storage & Peripherals', 'Biotechnology', 'Health Care Equipment', 'Property & Casualty Insurance']

RF top10: ['ABBV', 'WDC', 'BIIB', 'TECH', 'DHR', 'INCY', 'WAT', 'ZBH', 'GILD', 'VRTX']

=== Weekly Top3 (Bear Market — Use with Caution) ===
WDC: RF 0.537 | LSTM 0.558
ABBV: RF 0.537 | LSTM 0.549
ZBH: RF 0.537 | LSTM 0.542
